# Tidyverse डेटा हेरफेर और संवारना (Data Wrangling)

क्रॉस-भाषा डेटा पाइपलाइन: Python डेटा डाउनलोड करता है → R इसे dplyr से संसाधित करता है → Python परिणामों की कल्पना करता है।

**SharedVFS** का प्रदर्शन करता है — साझा फ़ाइल सिस्टम जो Python और R को फ़ाइलों का आदान-प्रदान करने देता है।

## 1. Python: डेटासेट डाउनलोड करें

In [ ]:
import micropip
await micropip.install('pandas')
import pandas as pd, pyodide.http, os

url = "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv"
resp = await pyodide.http.pyfetch(url)
text = await resp.string()

os.makedirs("/shared/data", exist_ok=True)
with open("/shared/data/gapminder.csv", "w") as f:
    f.write(text)

df = pd.read_csv("/shared/data/gapminder.csv")
print(f"Downloaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. R: dplyr + tidyr स्थापित करें और साझा डेटा पढ़ें

In [ ]:
install.packages(c("dplyr", "tidyr"))
library(dplyr)

gap <- read.csv("/shared/data/gapminder.csv")
cat("Read from SharedVFS:", nrow(gap), "rows\n")
glimpse(gap)

## 3. dplyr: महाद्वीप के अनुसार सारांश (2007)

In [ ]:
gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    countries = n(),
    mean_life = round(mean(lifeExp), 1),
    median_gdp = round(median(gdpPercap), 0),
    total_pop = sum(as.numeric(pop))
  ) %>%
  arrange(desc(mean_life))

## 4. dplyr: जीवन प्रत्याशा में सबसे बड़ा लाभ

In [ ]:
gains <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  select(country, continent, year, lifeExp) %>%
  tidyr::pivot_wider(names_from = year, values_from = lifeExp,
                     names_prefix = "y") %>%
  mutate(gain = y2007 - y1952) %>%
  arrange(desc(gain)) %>%
  head(10)
gains

## 5. dplyr: महाद्वीप के अनुसार जनसंख्या वृद्धि

In [ ]:
pop_growth <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  group_by(continent, year) %>%
  summarize(total_pop = sum(as.numeric(pop)), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = year, values_from = total_pop,
                     names_prefix = "pop_") %>%
  mutate(growth_pct = round((pop_2007 / pop_1952 - 1) * 100, 1)) %>%
  arrange(desc(growth_pct))
pop_growth

## 6. R: परिणामों को SharedVFS में लिखें

In [ ]:
# Write continent summary for Python to visualize
summary_2007 <- gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    mean_life = round(mean(lifeExp), 1),
    mean_gdp = round(mean(gdpPercap), 0),
    .groups = "drop"
  )
write.csv(summary_2007, "/shared/data/r_summary.csv", row.names = FALSE)
cat("Wrote /shared/data/r_summary.csv\n")
summary_2007

## 7. Python: R के परिणामों की कल्पना करें

In [ ]:
import micropip
await micropip.install('plotly')
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

r_summary = pd.read_csv("/shared/data/r_summary.csv")
print("Read from SharedVFS (written by R):")
print(r_summary.to_string(index=False))

fig = px.bar(r_summary, x="continent", y="mean_life",
             title="Mean Life Expectancy by Continent (2007) — from R → Python",
             labels={"mean_life": "Life Expectancy (years)", "continent": "Continent"},
             color="continent")
fig.update_layout(template='plotly_dark', showlegend=False)
show_plotly(fig)

In [ ]:
fig = px.scatter(r_summary, x="mean_gdp", y="mean_life",
                 text="continent", size=[40]*len(r_summary),
                 title="GDP vs Life Expectancy by Continent (R summary → Python plot)",
                 labels={"mean_gdp": "Mean GDP per Capita", "mean_life": "Mean Life Expectancy"})
fig.update_traces(textposition="top center")
fig.update_layout(template='plotly_dark')
fig.update_yaxes(range=[r_summary['mean_life'].min() - 2, r_summary['mean_life'].max() + 6])
show_plotly(fig)

## मुख्य बिंदु

- **Python** ने CSV डेटा को `/shared/data/` में डाउनलोड किया
- **R** ने इसे SharedVFS के माध्यम से पढ़ा और dplyr पाइपलाइन के साथ संसाधित किया
- **R** ने सारांश वापस `/shared/data/r_summary.csv` में लिखा
- **Python** ने R का आउटपुट पढ़ा और Plotly के साथ इंटरैक्टिव चार्ट बनाए

सभी फ़ाइल साझाकरण SharedVFS के माध्यम से होते हैं — कोई मैन्युअल आयात या निर्यात की आवश्यकता नहीं है।